# 02. One-Hot 인코딩 실험

베이스라인(#00, LabelEncoder 방식, CV MAE 0.2117)과 동일한 결측치 처리 + 파생변수 17개를 사용하고,
나머지 5개 명목형 컬럼(`gender`, `smoke_status`, `medical_history`, `family_medical_history`, `sleep_pattern`)만
**LabelEncoder 대신 One-Hot 인코딩**으로 바꿔서 CV 점수 변화를 확인합니다.

`activity`, `edu_level`은 순서형이라 #00과 동일하게 유지합니다.

In [1]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42  # 그라운드룰 1: 항상 42로 고정

## 1. Data Load

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print('train:', train.shape, '/ test:', test.shape)

train: (3000, 18) / test: (3000, 17)


## 2. 결측치 및 중복행 처리 (0909 ver, 팀 확정본)

In [3]:
# 중복 행 제거 (ID 제외 기준) - train에만 적용, test는 제거하지 않음
train = train.drop_duplicates(
    subset=[col for col in train.columns if col not in ['ID']]
).reset_index(drop=True)

# 근로시간 결측치: 0 처리
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

# 범주형 결측치: 독립 범주 신설
for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

print('결측치 처리 후 남은 결측 개수 - train:', train.isnull().sum().sum(), '/ test:', test.isnull().sum().sum())
print('중복 제거 후 train shape:', train.shape)

결측치 처리 후 남은 결측 개수 - train: 0 / test: 0
중복 제거 후 train shape: (2994, 18)


## 3. 파생변수 생성 (0909 ver, 팀 확정본 17개)

**중요**: 반드시 2단계(결측치 fillna)가 끝난 뒤, 4단계(인코딩) 이전에 실행해야 합니다.

In [4]:
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)

    # 1. 과로 및 생활 리듬
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)

    # 2. 질환 및 유전력
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)

    # 3. 심혈관 및 신체
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)

    # 4. 대사 및 노화
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)

    return data

train = add_features(train)
test = add_features(test)

print('파생변수 추가 후 train shape:', train.shape)

파생변수 추가 후 train shape: (2994, 35)


## 4. 인코딩 (One-Hot 버전)

- `activity`, `edu_level`: #00과 동일하게 순서형(Ordinal) 유지
- 나머지 5개(`gender`, `smoke_status`, `medical_history`, `family_medical_history`, `sleep_pattern`): **One-Hot 인코딩**

In [5]:
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}

train['activity'] = train['activity'].map(activity_map)
test['activity'] = test['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
test['edu_level'] = test['edu_level'].map(edu_map)

nominal_cols = ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']

# train/test를 합쳐서 One-Hot (카테고리 값 불일치 방지) 후 다시 분리
combined = pd.concat([train[nominal_cols], test[nominal_cols]], axis=0)
combined_encoded = pd.get_dummies(combined, columns=nominal_cols)

train_encoded = combined_encoded.iloc[:len(train)].reset_index(drop=True)
test_encoded = combined_encoded.iloc[len(train):].reset_index(drop=True)

train = train.drop(columns=nominal_cols).reset_index(drop=True)
test = test.drop(columns=nominal_cols).reset_index(drop=True)

train = pd.concat([train, train_encoded], axis=1)
test = pd.concat([test, test_encoded], axis=1)

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score']
x_test = test.drop('ID', axis=1)

print('One-Hot 적용 후 x_train:', x_train.shape, '(#00 LabelEncoder 방식보다 컬럼 수 늘어남)')

One-Hot 적용 후 x_train: (2994, 44) (#00 LabelEncoder 방식보다 컬럼 수 늘어남)


## 5. 5-Fold CV로 점수 측정 (#00과 비교)

In [7]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
val_maes = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(x_train), 1):
    X_tr, X_val = x_train.iloc[tr_idx], x_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = LGBMRegressor(random_state=RANDOM_STATE, verbose=-1)
    model.fit(X_tr, y_tr)

    val_pred = model.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_pred)
    val_maes.append(val_mae)
    print(f'fold {fold}: val MAE = {val_mae:.4f}')

print()
print(f'=== 5-Fold CV 평균 MAE (One-Hot, #01): {np.mean(val_maes):.4f} (+/- {np.std(val_maes):.4f}) ===')

fold 1: val MAE = 0.2130
fold 2: val MAE = 0.2053
fold 3: val MAE = 0.2154
fold 4: val MAE = 0.2152
fold 5: val MAE = 0.2066

=== 5-Fold CV 평균 MAE (One-Hot, #01): 0.2111 (+/- 0.0043) ===
